In [ ]:
# ============================================================================
# PART 1: Factorization Methods
# ============================================================================

def get_factors_for_layer(grads_dir: Path, layer_name: str, 
                          start_batch: int = None, end_batch: int = None,
                          topk: int = 1) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Extract Kronecker factors XF, YF from saved gradients for a specific layer.
    
    Args:
        grads_dir: Directory containing .safetensors gradient files
        layer_name: Name of the layer (e.g., "fc1.weight")
        start_batch: Starting batch index (None = use all)
        end_batch: Ending batch index (None = use all)
        topk: Number of top singular values to compute
        
    Returns:
        XF: (in_features, in_features) array
        YF: (out_features, out_features) array
        s: (topk,) singular values
    """
    bylayer, files = load_grads_by_layer(grads_dir, recursive=False)
    
    if layer_name not in bylayer:
        raise ValueError(f"Layer {layer_name} not found. Available: {list(bylayer.keys())}")
    
    grads_list = bylayer[layer_name]
    
    # Use specified range or all gradients
    if start_batch is not None or end_batch is not None:
        grads_list = grads_list[start_batch:end_batch]
    
    XF, YF, s = get_kron_factors_worker_cpu(
        grads_list, 
        topk=topk,
        dtype=np.float32,
        verbose=True
    )
    
    return XF, YF, s


def factorize_layer_kron_method(layer: nn.Linear, XF: np.ndarray, YF: np.ndarray,
                                 rank: int, **kwargs) -> nn.Sequential:
    """
    Factorize layer using Kronecker-factored SVD (FastKron-like, with correlations).
    
    Args:
        layer: Original nn.Linear layer
        XF: Input-side Kronecker factor
        YF: Output-side Kronecker factor
        rank: Target rank for compression
        **kwargs: Additional args for factorize_layer_kron_svd
        
    Returns:
        nn.Sequential containing two factorized linear layers
    """
    return factorize_layer_kron_svd(
        layer=layer,
        XF=XF,
        YF=YF,
        rank=rank,
        **kwargs
    )


def factorize_layer_diag_method(layer: nn.Linear, XF: np.ndarray, YF: np.ndarray,
                                 rank: int, **kwargs) -> nn.Sequential:
    """
    Factorize using only diagonal of Kronecker factors (FWSVD-like, no correlations).
    
    Args:
        layer: Original nn.Linear layer
        XF: Input-side Kronecker factor (only diagonal will be used)
        YF: Output-side Kronecker factor (only diagonal will be used)
        rank: Target rank for compression
        **kwargs: Additional args for factorize_layer_kron_svd
        
    Returns:
        nn.Sequential containing two factorized linear layers
    """
    # Extract only diagonal elements
    XF_diag = np.diag(np.diag(XF))
    YF_diag = np.diag(np.diag(YF))
    
    return factorize_layer_kron_svd(
        layer=layer,
        XF=XF_diag,
        YF=YF_diag,
        rank=rank,
        **kwargs
    )


def factorize_layer_svd_method(layer: nn.Linear, rank: int, **kwargs) -> nn.Sequential:
    """
    Factorize using plain SVD (no Kronecker structure).
    
    Args:
        layer: Original nn.Linear layer
        rank: Target rank for compression
        **kwargs: Additional args for factorize_layer_kron_svd
        
    Returns:
        nn.Sequential containing two factorized linear layers
    """
    out_features, in_features = layer.weight.shape
    
    # Use identity matrices as factors (no whitening)
    XF_identity = np.eye(in_features, dtype=np.float32)
    YF_identity = np.eye(out_features, dtype=np.float32)
    
    return factorize_layer_kron_svd(
        layer=layer,
        XF=XF_identity,
        YF=YF_identity,
        rank=rank,
        **kwargs
    )


# ============================================================================
# PART 2: Model Manipulation & Evaluation
# ============================================================================

def replace_layer_in_model(model: nn.Module, layer_name: str, 
                           new_layer: nn.Module) -> nn.Module:
    """
    Replace a layer in model with new_layer.
    
    Args:
        model: The model to modify
        layer_name: Name of layer to replace (e.g., "fc1")
        new_layer: New layer (can be nn.Sequential)
        
    Returns:
        Modified model (in-place modification)
    """
    # Handle nested names like "encoder.fc1"
    parts = layer_name.split('.')
    parent = model
    
    for part in parts[:-1]:
        parent = getattr(parent, part)
    
    setattr(parent, parts[-1], new_layer)
    return model


def evaluate_model(model: nn.Module, loader: DataLoader, 
                   device: torch.device = torch.device('cpu')) -> Dict[str, float]:
    """
    Evaluate model accuracy and loss on a dataset.
    
    Args:
        model: Model to evaluate
        loader: DataLoader for evaluation
        device: Device to use
        
    Returns:
        Dictionary with 'accuracy' and 'loss' keys
    """
    model.eval()
    model.to(device)
    
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0
    
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            
            total_loss += loss.item()
            total_acc += accuracy(logits, yb)
            n_batches += 1
    
    return {
        'accuracy': total_acc / max(1, n_batches),
        'loss': total_loss / max(1, n_batches)
    }


def count_parameters(model: nn.Module) -> int:
    """Count total trainable parameters in model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================================
# PART 3: Compression Experiment Runner
# ============================================================================

def compress_and_evaluate(model: nn.Module, 
                         layer_name: str,
                         method: str,
                         rank: int,
                         val_loader: DataLoader,
                         XF: np.ndarray = None,
                         YF: np.ndarray = None,
                         device: torch.device = torch.device('cpu')) -> Dict[str, Any]:
    """
    Compress a specific layer using given method and evaluate.
    
    Args:
        model: Original model (will be deep-copied)
        layer_name: Name of layer to compress (e.g., "fc1")
        method: One of {'kron_svd', 'diag_kron', 'plain_svd'}
        rank: Compression rank
        val_loader: Validation DataLoader
        XF: Input Kronecker factor (required for kron_svd, diag_kron)
        YF: Output Kronecker factor (required for kron_svd, diag_kron)
        device: Device for evaluation
        
    Returns:
        Dictionary with evaluation metrics + metadata
    """
    import copy
    
    # Deep copy model to avoid modifying original
    model_compressed = copy.deepcopy(model)
    
    # Get original layer
    parts = layer_name.split('.')
    original_layer = model_compressed
    for part in parts:
        original_layer = getattr(original_layer, part)
    
    if not isinstance(original_layer, nn.Linear):
        raise ValueError(f"Layer {layer_name} is not nn.Linear")
    
    # Factorize using appropriate method
    if method == 'kron_svd':
        if XF is None or YF is None:
            raise ValueError("XF and YF required for kron_svd method")
        factorized_layer = factorize_layer_kron_method(original_layer, XF, YF, rank)
        
    elif method == 'diag_kron':
        if XF is None or YF is None:
            raise ValueError("XF and YF required for diag_kron method")
        factorized_layer = factorize_layer_diag_method(original_layer, XF, YF, rank)
        
    elif method == 'plain_svd':
        factorized_layer = factorize_layer_svd_method(original_layer, rank)
        
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Replace layer in model
    replace_layer_in_model(model_compressed, layer_name, factorized_layer)
    
    # Evaluate
    metrics = evaluate_model(model_compressed, val_loader, device)
    
    # Add metadata
    metrics.update({
        'method': method,
        'rank': rank,
        'layer': layer_name,
        'params_compressed': count_parameters(model_compressed)
    })
    
    return metrics


# ============================================================================
# PART 4: Full Experiment Pipeline
# ============================================================================

def run_compression_experiment(
    ds_name: str,
    ds_class: type,
    model_class: type,
    target_layer: str,
    ranks: List[int],
    grads_dir: Path,
    n_samples: int = 2000,
    in_dim: int = 20,
    hidden_dim: int = 32,
    batch_size: int = 32,
    val_split: float = 0.2,
    device: torch.device = torch.device('cpu'),
    use_last_n_batches: int = None,
) -> pd.DataFrame:
    """
    Run full compression experiment for one dataset.
    
    Pipeline:
    1. Load trained model
    2. Extract Kronecker factors from gradients
    3. For each rank and each method:
       - Compress target layer
       - Evaluate on validation set
    4. Return results as DataFrame
    
    Args:
        ds_name: Dataset name (for labeling)
        ds_class: Dataset class (IsotropicDataset, etc.)
        model_class: Model class (OneLayerMLP, TwoLayerMLP)
        target_layer: Layer name to compress (e.g., "fc1")
        ranks: List of ranks to test
        grads_dir: Directory containing saved gradients
        n_samples: Total samples in dataset
        in_dim: Input dimension
        hidden_dim: Hidden dimension (if applicable)
        batch_size: Batch size for evaluation
        val_split: Validation split ratio
        device: Device for evaluation
        use_last_n_batches: Use only last N batches for factor estimation (None = use all)
        
    Returns:
        DataFrame with columns: dataset, method, rank, accuracy, loss, params_compressed, layer
    """
    print(f"\n{'='*60}")
    print(f"Running experiment: {ds_name}")
    print(f"{'='*60}")
    
    # 1. Create dataset and split
    ds = ds_class(n_samples=n_samples, in_dim=in_dim, seed=42)
    X, y = ds.create_data()
    X = torch.as_tensor(X, dtype=torch.float32)
    
    # Train/val split
    n_val = int(n_samples * val_split)
    n_train = n_samples - n_val
    
    X_train, X_val = X[:n_train], X[n_train:]
    y_train, y_val = y[:n_train], y[n_train:]
    
    val_dataset = TensorDataset(X_val, y_val)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # 2. Load/create model (assuming already trained)
    # If using OneLayerMLP
    if 'hidden' in str(model_class.__name__).lower() or model_class == TwoLayerMLP:
        model = model_class(in_dim=in_dim, hidden_dim=hidden_dim, out_dim=2)
    else:
        model = model_class(in_dim=in_dim, out_dim=2)
    
    # NOTE: You should load trained weights here if you saved them
    # For now, we assume the model in memory is already trained
    # model.load_state_dict(torch.load(...))
    
    # 3. Get baseline accuracy
    baseline_metrics = evaluate_model(model, val_loader, device)
    print(f"Baseline accuracy: {baseline_metrics['accuracy']:.4f}")
    
    # 4. Extract Kronecker factors from gradients
    print(f"Extracting Kronecker factors for layer: {target_layer}")
    
    # Determine batch range
    bylayer, files = load_grads_by_layer(grads_dir, recursive=False)
    total_batches = len(files)
    
    if use_last_n_batches is not None:
        start_batch = max(0, total_batches - use_last_n_batches)
        end_batch = total_batches
        print(f"Using batches {start_batch} to {end_batch} (last {use_last_n_batches})")
    else:
        start_batch = 0
        end_batch = total_batches
        print(f"Using all {total_batches} batches")
    
    # Get factors
    layer_key = f"{target_layer}.weight"  # Gradients are saved with .weight suffix
    XF, YF, s = get_factors_for_layer(
        grads_dir, 
        layer_key, 
        start_batch=start_batch,
        end_batch=end_batch,
        topk=2
    )
    print(f"Factors extracted: XF {XF.shape}, YF {YF.shape}, top-2 σ: {s}")
    
    # 5. Run compression experiments
    results = []
    methods = ['kron_svd', 'diag_kron', 'plain_svd']
    
    for rank in ranks:
        print(f"\n  Testing rank={rank}")
        for method in methods:
            print(f"    Method: {method}...", end=' ')
            
            metrics = compress_and_evaluate(
                model=model,
                layer_name=target_layer,
                method=method,
                rank=rank,
                val_loader=val_loader,
                XF=XF,
                YF=YF,
                device=device
            )
            
            metrics['dataset'] = ds_name
            metrics['baseline_acc'] = baseline_metrics['accuracy']
            results.append(metrics)
            
            print(f"acc={metrics['accuracy']:.4f}")
    
    return pd.DataFrame(results)


# ============================================================================
# PART 5: Visualization
# ============================================================================

def plot_compression_results(df: pd.DataFrame, save_path: Path = None):
    """
    Plot compression results: accuracy vs rank for different methods and datasets.
    
    Args:
        df: DataFrame from run_compression_experiment
        save_path: Optional path to save figure
    """
    import seaborn as sns
    import matplotlib.pyplot as plt
    
    datasets = df['dataset'].unique()
    n_datasets = len(datasets)
    
    fig, axes = plt.subplots(1, n_datasets, figsize=(6*n_datasets, 4), squeeze=False)
    axes = axes.flatten()
    
    sns.set_theme(style='whitegrid', context='talk')
    
    # Style mapping for methods
    style_map = {
        'kron_svd': '-',      # solid
        'diag_kron': '--',    # dashed
        'plain_svd': ':',     # dotted
    }
    
    for idx, ds_name in enumerate(datasets):
        ax = axes[idx]
        df_ds = df[df['dataset'] == ds_name]
        
        # Plot lines for each method
        for method in df_ds['method'].unique():
            df_method = df_ds[df_ds['method'] == method]
            df_method = df_method.sort_values('rank')
            
            ax.plot(
                df_method['rank'], 
                df_method['accuracy'],
                label=method,
                linestyle=style_map.get(method, '-'),
                marker='o',
                linewidth=2,
                markersize=6
            )
        
        # Plot baseline
        baseline_acc = df_ds['baseline_acc'].iloc[0]
        ax.axhline(baseline_acc, color='gray', linestyle='-.', 
                   linewidth=1.5, label='baseline', alpha=0.7)
        
        ax.set_title(f'{ds_name}')
        ax.set_xlabel('Rank')
        ax.set_ylabel('Accuracy')
        ax.legend(frameon=True)
        ax.grid(True, alpha=0.3)
    
    fig.tight_layout()
    
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved figure to {save_path}")
    
    return fig


# ============================================================================
# PART 6: Main Execution
# ============================================================================

def run_all_experiments():
    """
    Main function to run all experiments across datasets.
    """
    # Configuration
    datasets = {
        'Isotropic': IsotropicDataset,
        'Anisotropic': AnisotropicDataset,
        'Correlated': CorrelatedToeplitzDataset,
    }
    
    ranks = [1, 2, 4, 8, 16]  # Adjust based on your layer size
    target_layer = 'fc1'  # or 'fc2' for TwoLayerMLP
    
    root_grads_dir = Path('./grads_dump_batches')
    results_all = []
    
    for ds_name, ds_class in datasets.items():
        grads_dir = root_grads_dir / ds_name.lower()
        
        df_results = run_compression_experiment(
            ds_name=ds_name,
            ds_class=ds_class,
            model_class=OneLayerMLP,  # or TwoLayerMLP
            target_layer=target_layer,
            ranks=ranks,
            grads_dir=grads_dir,
            use_last_n_batches=30,  # Use last epoch if epochs=30, batches_per_epoch~60
        )
        
        results_all.append(df_results)
    
    # Combine all results
    df_all = pd.concat(results_all, ignore_index=True)
    
    # Save results
    df_all.to_csv('compression_results.csv', index=False)
    print("\nResults saved to compression_results.csv")
    
    # Plot
    fig = plot_compression_results(df_all, save_path=Path('compression_plot.png'))
    plt.show()
    
    return df_all
